# Why is d'(ISI=1) vs sigma1 Non-Monotonic?

The position-bias notebook showed that for single-ISI conditions, positions
are **already uniform** (mean=0.51, KS p>0.7). So position bias is NOT the
root cause.

This notebook digs into the **model scoring mechanism** to find the real cause.

| Section | Test |
|---|---|
| 1 | d' decomposition: mean hit score & FA score vs sigma1 |
| 2 | Hit/FA score distributions at low, mid, high sigma1 |
| 3 | Noise schedule visualization: what drift does ISI=1 actually get? |
| 4 | Target vs nearest-non-target: which dominates hit scores? |
| 5 | MC convergence: is the bump real or sampling noise? |
| 6 | Bank size controlled: fix bank size, vary sigma1 |

In [ ]:
import sys, os, yaml, torch, random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter, defaultdict

sys.path.append('/om2/user/jmhicks/projects/TextureStreaming/code/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/utls/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/src/model/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/')

from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params, statistics_set, texture_dataset
from texture_prior.utils import path

from utls.runners_v2 import run_experiment_scores, make_noise_schedule, ThreeRegimeNoise
from utls.runners_utils import *
from utls.analysis_helpers import auroc_to_dprime
from utls.toy_experiments import (
    make_compact_multi_isi_sequences,
    infer_trial_isis,
)
from utls.sigma_fitting import log_mid
from encoders import *

from sklearn.metrics import roc_auc_score, roc_curve

try:
    get_ipython()
    from tqdm.notebook import tqdm, trange
except NameError:
    from tqdm import tqdm, trange

In [ ]:
# --- Load config, experiment data, and encode stimuli ---
# (Same setup as the deep-dive notebook)

def load_config(cfg_path):
    cfg_path = Path(cfg_path)
    if not cfg_path.exists():
        raise FileNotFoundError(cfg_path)
    with open(cfg_path) as f:
        return yaml.safe_load(f), cfg_path

CONFIG_PATH = (
    "/om2/user/bjmedina/auditory-memory/memory/"
    "model_yamls/three-regime/resnet50/nontime_avg/run_000005.yaml"
)
model_cfg, _ = load_config(CONFIG_PATH)

exp_cfg = model_cfg["experiment"]
which_task, is_multi = exp_cfg["which_task"], exp_cfg["is_multi"]
which_isi = exp_cfg.get("which_isi", None)
isis = [0, 1, 2, 4, 8, 16, 32, 64] if is_multi else [0, which_isi]

metric = model_cfg["metric"]
noise_cfg = model_cfg["noise_model"]
noise_mode, t_step = noise_cfg["name"], noise_cfg["t_step"]

repr_cfg = model_cfg["representation"]
time_avg = repr_cfg["time_avg"]
encoder_type = repr_cfg["type"]
layer = repr_cfg.get("layer", None)
pc_dims = repr_cfg.get("pc_dims", None)

exp_list, all_files, name_to_idx, human_runs, task_name, hr_task_name = (
    load_experiment_data(which_task, which_isi, is_multi, old=False)
)
human_curve = compute_human_curve(human_runs, is_multi, which_isi)

# Encode stimuli
NN_ENCODERS = {"kell2018", "resnet50"}
encoder_task = "word_speaker_audioset" if encoder_type in NN_ENCODERS else "audioset"
encoder_cfg = dict(
    encoder_type=encoder_type, model_name=encoder_type, task=encoder_task,
    statistics_dict=statistics_set.statistics, model_params=model_params,
    pc_dims=pc_dims, sr=20000, duration=2.0, rms_level=0.05,
    time_avg=time_avg, device="cuda",
)
if encoder_type in NN_ENCODERS:
    encoder_cfg["layer"] = layer
encoder = build_encoder(encoder_cfg)
X = encode_stimuli(encoder, all_files)

# Param bounds and human targets
param_bounds = {"sigma0": (3.0, 22), "sigma1": (0.01, 10.0), "sigma2": (0.0005, 0.5)}
sigma1_human = {isi: float(human_curve[i]) for i, isi in enumerate(isis) if isi in [1, 2, 4]}
sigma2_init = log_mid(*param_bounds["sigma2"])

stimulus_pool = sorted({s for seq in exp_list for s in seq})

print(f"ISIs: {isis}")
print(f"Human d' curve: {human_curve}")
print(f"Stimulus pool: {len(stimulus_pool)} items")
print(f"Encoded shape: {X.shape}")
print(f"sigma1 human targets: {sigma1_human}")
print(f"noise_mode: {noise_mode}, t_step: {t_step}")

In [ ]:
# Generate the same sequences as Investigation C
e_short, k_short = make_compact_multi_isi_sequences(
    stimulus_pool=stimulus_pool, isi_values=[1],
    n_sequences=8, length=15, min_pairs_per_isi=5, seed=1000,
)

e_long, k_long = make_compact_multi_isi_sequences(
    stimulus_pool=stimulus_pool, isi_values=[1],
    n_sequences=10, length=60, min_pairs_per_isi=5, seed=1001,
)

e_mixed, k_mixed = make_compact_multi_isi_sequences(
    stimulus_pool=stimulus_pool, isi_values=[1, 2, 4],
    n_sequences=10, length=60, min_pairs_per_isi=5, seed=3000,
)

seq_configs = {
    'ISI=[1] L=15': {'exps': e_short, 'keys': k_short},
    'ISI=[1] L=60': {'exps': e_long,  'keys': k_long},
    'ISI=[1,2,4] L=60': {'exps': e_mixed, 'keys': k_mixed},
}

SIGMA0 = 13.5  # typical fitted value
SIGMA1_GRID = np.exp(np.linspace(np.log(0.01), np.log(10.0), 30))

for name, cfg in seq_configs.items():
    print(f"{name}: {len(cfg['exps'])} seqs x {len(cfg['exps'][0])} trials")
print(f"\nSIGMA1 grid: {len(SIGMA1_GRID)} points [{SIGMA1_GRID[0]:.4f} .. {SIGMA1_GRID[-1]:.4f}]")

---
## 1. d' Decomposition: Mean Hit & FA Scores vs sigma1

If d' is non-monotonic, it's because the **gap** between hit and FA score
distributions changes non-monotonically. Let's plot the raw scores.

In [ ]:
# Run sigma1 sweep collecting raw hit and FA score arrays

def sweep_sigma1_decomposed(
    experiment_list, sigma0, sigma1_grid,
    n_mc=64, desc="sweep", seed=0
):
    """For each sigma1, collect raw hit/FA scores and d'."""
    seq_trial_isis = [infer_trial_isis(seq) for seq in experiment_list]
    results = []

    for gi in trange(len(sigma1_grid), desc=desc):
        s1 = sigma1_grid[gi]
        all_hits_isi1 = []
        all_fas = []
        all_fa_bank_sizes = []  # bank size at each FA
        all_hit_bank_sizes = []  # bank size at each ISI=1 hit

        for rep in range(n_mc):
            for si, seq in enumerate(experiment_list):
                t_isis = seq_trial_isis[si]
                run_out = run_experiment_scores(
                    sigma0=sigma0, sigma1=s1, sigma2=sigma2_init,
                    t_step=t_step, rate=0, noise_mode=noise_mode,
                    metric=metric, X0=X, name_to_idx=name_to_idx,
                    experiment_list=[seq], debug=False,
                    seed=seed + rep * 1000 + si + gi * 100_000,
                )
                h = np.asarray(run_out["hits"])
                f = np.asarray(run_out["fas"])

                if len(h) != len(t_isis):
                    continue

                # ISI=1 hits
                mask = np.array(t_isis) == 1
                all_hits_isi1.extend(h[mask].tolist())

                # FA scores with bank-size info from fa_by_t
                all_fas.extend(f.tolist())

                # Bank sizes from isi_hit_dists (key=2 for ISI=1)
                for score_val, trial_pos in run_out["isi_hit_dists"].get(2, []):
                    all_hit_bank_sizes.append(trial_pos)

                for t_pos, fa_list in enumerate(run_out["fa_by_t"]):
                    for score_val in fa_list:
                        all_fa_bank_sizes.append(t_pos)

        hits_arr = np.array(all_hits_isi1)
        fas_arr = np.array(all_fas)

        # Compute d'
        if len(hits_arr) > 0 and len(fas_arr) > 0:
            y_true = np.concatenate([np.ones(len(hits_arr)), np.zeros(len(fas_arr))])
            scores = np.concatenate([hits_arr, fas_arr])
            auroc = roc_auc_score(y_true, -scores)
            dprime = auroc_to_dprime(auroc)
        else:
            auroc, dprime = np.nan, np.nan

        results.append({
            'sigma1': s1,
            'dprime': dprime,
            'auroc': auroc,
            'hit_mean': np.mean(hits_arr) if len(hits_arr) > 0 else np.nan,
            'hit_std': np.std(hits_arr) if len(hits_arr) > 0 else np.nan,
            'hit_median': np.median(hits_arr) if len(hits_arr) > 0 else np.nan,
            'fa_mean': np.mean(fas_arr) if len(fas_arr) > 0 else np.nan,
            'fa_std': np.std(fas_arr) if len(fas_arr) > 0 else np.nan,
            'fa_median': np.median(fas_arr) if len(fas_arr) > 0 else np.nan,
            'n_hits': len(hits_arr),
            'n_fas': len(fas_arr),
            'hit_scores': hits_arr,
            'fa_scores': fas_arr,
        })

    return results


N_MC = 64
sweeps = {}
for name, cfg in seq_configs.items():
    print(f"\n--- {name} ---")
    sweeps[name] = sweep_sigma1_decomposed(
        cfg['exps'], sigma0=SIGMA0, sigma1_grid=SIGMA1_GRID,
        n_mc=N_MC, desc=name, seed=700_000,
    )

In [ ]:
# KEY PLOT: d', mean hit score, mean FA score, and the gap — all vs sigma1

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for name, results in sweeps.items():
    s1 = np.array([r['sigma1'] for r in results])
    dp = np.array([r['dprime'] for r in results])
    hm = np.array([r['hit_mean'] for r in results])
    fm = np.array([r['fa_mean'] for r in results])
    hs = np.array([r['hit_std'] for r in results])
    fs = np.array([r['fa_std'] for r in results])
    gap = fm - hm  # for distance metric: larger gap = better discrimination

    axes[0, 0].plot(s1, dp, 'o-', ms=3, label=name)
    axes[0, 1].plot(s1, hm, 'o-', ms=3, label=name)
    axes[0, 2].plot(s1, fm, 'o-', ms=3, label=name)
    axes[1, 0].plot(s1, gap, 'o-', ms=3, label=name)
    axes[1, 1].plot(s1, hs, 'o-', ms=3, label=name)
    axes[1, 2].plot(s1, fs, 'o-', ms=3, label=name)

titles = ["d'(ISI=1)", "Mean hit score", "Mean FA score",
          "Gap (FA_mean - hit_mean)", "Hit score std", "FA score std"]
for ax, title in zip(axes.flat, titles):
    ax.set_xscale('log')
    ax.set_xlabel('sigma1')
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=7, frameon=False)
    ax.grid(alpha=0.25)

axes[0, 0].axhline(sigma1_human.get(1, 0), ls=':', color='red', alpha=0.7, label="human d'")

fig.suptitle("Section 1: d' decomposition into hit/FA score components\n"
             "Non-monotonic d' requires gap to be non-monotonic",
             y=1.03, fontsize=13)
fig.tight_layout()
plt.show()

---
## 2. Hit/FA Score Distributions

Visualize the full score distributions at low, mid, and high sigma1
to see how they overlap.

In [ ]:
# Pick 3 sigma1 values: low, bump-peak (if exists), high
# Use the ISI=[1] L=15 sweep to find peak

results_short = sweeps['ISI=[1] L=15']
dp_arr = np.array([r['dprime'] for r in results_short])
peak_idx = np.nanargmax(dp_arr)

idx_low = 2  # near the start
idx_mid = peak_idx  # at the d' peak
idx_high = len(SIGMA1_GRID) - 3  # near the end

print(f"Selected sigma1 values:")
print(f"  low:  sigma1 = {SIGMA1_GRID[idx_low]:.4f} (d' = {dp_arr[idx_low]:.3f})")
print(f"  peak: sigma1 = {SIGMA1_GRID[idx_mid]:.4f} (d' = {dp_arr[idx_mid]:.3f})")
print(f"  high: sigma1 = {SIGMA1_GRID[idx_high]:.4f} (d' = {dp_arr[idx_high]:.3f})")

diag_indices = [idx_low, idx_mid, idx_high]
diag_labels = ['low sigma1', 'peak sigma1', 'high sigma1']

fig, axes = plt.subplots(len(sweeps), 3, figsize=(18, 4 * len(sweeps)))
if len(sweeps) == 1:
    axes = axes[np.newaxis, :]

for row, (name, results) in enumerate(sweeps.items()):
    for col, (idx, label) in enumerate(zip(diag_indices, diag_labels)):
        ax = axes[row, col]
        r = results[idx]
        s1 = r['sigma1']

        hits = r['hit_scores']
        fas = r['fa_scores']

        all_scores = np.concatenate([hits, fas])
        lo, hi = np.percentile(all_scores, [1, 99])
        bins = np.linspace(lo, hi, 50)

        ax.hist(fas, bins=bins, alpha=0.5, density=True, color='tab:blue', label='FA')
        ax.hist(hits, bins=bins, alpha=0.5, density=True, color='tab:orange', label='Hit (ISI=1)')

        ax.axvline(np.mean(fas), color='tab:blue', ls='--', alpha=0.7)
        ax.axvline(np.mean(hits), color='tab:orange', ls='--', alpha=0.7)

        ax.set_title(f"{label}: sigma1={s1:.3f}\nd'={r['dprime']:.3f}", fontsize=10)
        ax.set_xlabel('Distance score')
        ax.legend(fontsize=7, frameon=False)
        ax.grid(alpha=0.25)

        if col == 0:
            ax.set_ylabel(f"{name}\nDensity")

fig.suptitle("Section 2: Score distributions at low / peak / high sigma1\n"
             "Hit should be LEFT of FA (lower distance). d' = separation.",
             y=1.03, fontsize=13)
fig.tight_layout()
plt.show()

---
## 3. Noise Schedule Visualization

For ISI=1, the target memory has age=2 at probe time.  
With `ThreeRegimeNoise(sigma0, sigma1, sigma2, t_step)`:  
- age=1 → drift = 1e-5 (essentially zero)  
- age=2 → drift = sigma1 (since 2 < t_step=5)  

So sigma1 **directly controls the drift noise on the ISI=1 target**.  
But it also controls drift on ALL memories with 1 < age < t_step.

In [ ]:
# Visualize the noise schedule for a range of sigma1 values

ages = np.arange(0, 20)
sigma1_examples = [0.01, 0.1, 0.5, 1.0, 3.0, 10.0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for s1 in sigma1_examples:
    schedule = ThreeRegimeNoise(SIGMA0, s1, sigma2_init, t_step)
    noise_at_age = [schedule(a) for a in ages]
    axes[0].plot(ages, noise_at_age, 'o-', ms=4, label=f'sigma1={s1}')

axes[0].axvline(2, ls=':', color='red', alpha=0.5, label='ISI=1 target age')
axes[0].axvline(t_step, ls='--', color='gray', alpha=0.5, label=f't_step={t_step}')
axes[0].set_xlabel('Memory age')
axes[0].set_ylabel('Drift noise std')
axes[0].set_title('Drift noise schedule by memory age')
axes[0].legend(fontsize=7, frameon=False)
axes[0].set_yscale('log')
axes[0].grid(alpha=0.25)

# Cumulative drift: total accumulated noise variance after N rounds
# var_total = sum(schedule(a)^2 for a in 1..age)
for s1 in sigma1_examples:
    schedule = ThreeRegimeNoise(SIGMA0, s1, sigma2_init, t_step)
    cum_var = []
    for age in ages:
        var = sum(schedule(a)**2 for a in range(1, age + 1))
        cum_var.append(np.sqrt(var))  # total std
    axes[1].plot(ages, cum_var, 'o-', ms=4, label=f'sigma1={s1}')

axes[1].axvline(2, ls=':', color='red', alpha=0.5, label='ISI=1 target age')
axes[1].axvline(t_step, ls='--', color='gray', alpha=0.5, label=f't_step={t_step}')
axes[1].set_xlabel('Memory age')
axes[1].set_ylabel('Cumulative drift std (sqrt sum of variances)')
axes[1].set_title('Total accumulated drift noise')
axes[1].legend(fontsize=7, frameon=False)
axes[1].grid(alpha=0.25)

fig.suptitle(f'Section 3: Noise schedule (sigma0={SIGMA0}, t_step={t_step})',
             y=1.03, fontsize=13)
fig.tight_layout()
plt.show()

print(f"\nFor ISI=1 target (age=2):")
print(f"  Encoding noise std = sigma0 * dim_std = {SIGMA0} * dim_std")
print(f"  Drift noise std at age=2 = sigma1")
print(f"  Drift at age=1 = {ThreeRegimeNoise(SIGMA0, 1.0, sigma2_init, t_step)(1):.1e} (≈ zero)")
print(f"  So total noise ≈ sqrt(sigma0² + sigma1²) * dim_std")

---
## 4. Target vs Nearest-Non-Target

The hit score = `min(distance to ALL memories)`. For ISI=1 hits, this is the
min of:
- Distance to the **target memory** (the matching first-pres, noisy)
- Distance to the **nearest non-target** in the bank

If the target is always closest (low sigma1), hit scores are dominated by the
target. If sigma1 is high enough that the target drifts far, the hit score
becomes dominated by a non-target → indistinguishable from FA → d' drops.

**Question**: at what sigma1 does the transition happen?

In [ ]:
# Modified model run that tracks target distance vs best-non-target distance

def run_with_target_tracking(
    experiment_list, sigma0, sigma1,
    n_mc=32, seed=0,
):
    """Run model and for each ISI=1 hit, record:
    - distance to the target memory
    - distance to the nearest non-target memory
    - which one wins (= actual hit score)
    """
    from utls.runners_v2 import (
        compute_score, make_noise_schedule, ThreeRegimeNoise
    )

    schedule = ThreeRegimeNoise(sigma0, sigma1, sigma2_init, t_step)

    device = X.device
    dtype = X.dtype
    dim_std = X.std(dim=0)
    rms_std = torch.sqrt(torch.mean(dim_std ** 2))
    scaled_std = dim_std / rms_std

    idx_to_name = {v: k for k, v in name_to_idx.items()}
    records = []  # (target_dist, nontarget_dist, hit_score, bank_size)

    for rep in range(n_mc):
        torch_rng = torch.Generator(device=device)
        torch_rng.manual_seed(seed + rep)

        for seq in experiment_list:
            seq_idx = [name_to_idx[f] for f in seq]
            memory_bank = []
            seen = {}

            for t, incoming in enumerate(seq_idx, start=1):
                probe = X[incoming].view(1, -1)

                # Update memories and compute scores
                target_score = None
                nontarget_scores = []

                for mem in memory_bank:
                    age = t - mem['t_inserted']
                    std = schedule(age)
                    noise = torch.randn(
                        mem['mu'].shape, device=device, dtype=dtype,
                        generator=torch_rng,
                    ) * (std * scaled_std)
                    mem['mu'] += noise

                    score = compute_score(probe, mem['mu'], std, metric)

                    if mem['stim_idx'] == incoming:
                        target_score = score
                    else:
                        nontarget_scores.append(score)

                # Record for ISI=1 repeats
                if incoming in seen:
                    isi = t - seen[incoming] - 1
                    if isi == 1 and target_score is not None and nontarget_scores:
                        best_nontarget = min(nontarget_scores)
                        actual_hit = min(target_score, best_nontarget)
                        records.append({
                            'target_dist': target_score,
                            'nontarget_dist': best_nontarget,
                            'hit_score': actual_hit,
                            'target_wins': target_score <= best_nontarget,
                            'bank_size': len(memory_bank),
                        })

                # Store memory
                base = X[incoming].clone()
                noise = torch.randn(
                    base.shape, device=device, dtype=dtype,
                    generator=torch_rng,
                ) * (sigma0 * dim_std)
                mem_new = base + noise
                memory_bank.append({
                    'mu': mem_new.view(1, -1),
                    't_inserted': t,
                    'stim_idx': incoming,
                })
                if incoming not in seen:
                    seen[incoming] = t

    return records


# Run at a few sigma1 values
sigma1_diag = [0.01, 0.05, 0.1, 0.3, 0.5, 1.0, 3.0, 10.0]

target_tracking = {}
for name, cfg in seq_configs.items():
    target_tracking[name] = {}
    for s1 in tqdm(sigma1_diag, desc=name):
        records = run_with_target_tracking(
            cfg['exps'], sigma0=SIGMA0, sigma1=s1,
            n_mc=32, seed=800_000 + int(s1 * 100),
        )
        target_tracking[name][s1] = records

print("Done.")

In [ ]:
# Plot: fraction of ISI=1 hits where target wins vs non-target wins

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, tracking) in zip(axes, target_tracking.items()):
    s1_vals = []
    frac_target_wins = []
    mean_target_dist = []
    mean_nontarget_dist = []

    for s1 in sigma1_diag:
        records = tracking[s1]
        if not records:
            continue
        s1_vals.append(s1)
        frac = np.mean([r['target_wins'] for r in records])
        frac_target_wins.append(frac)
        mean_target_dist.append(np.mean([r['target_dist'] for r in records]))
        mean_nontarget_dist.append(np.mean([r['nontarget_dist'] for r in records]))

    ax2 = ax.twinx()
    ax.plot(s1_vals, frac_target_wins, 'o-', color='tab:green', ms=6,
            label='Frac. target wins', linewidth=2)
    ax2.plot(s1_vals, mean_target_dist, 's--', color='tab:orange', ms=5,
             label='Mean target dist')
    ax2.plot(s1_vals, mean_nontarget_dist, 's--', color='tab:blue', ms=5,
             label='Mean non-target dist')

    ax.set_xscale('log')
    ax.set_xlabel('sigma1')
    ax.set_ylabel('Fraction target wins (green)', color='tab:green')
    ax.set_ylim(-0.05, 1.05)
    ax.set_title(name, fontsize=11)
    ax.grid(alpha=0.25)
    ax2.set_ylabel('Mean distance', color='tab:blue')

    # Combined legend
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, fontsize=7, frameon=False, loc='center right')

fig.suptitle("Section 4: Is the hit score from the TARGET or the nearest NON-TARGET?\n"
             "When target_wins fraction drops, hits become indistinguishable from FAs",
             y=1.05, fontsize=13)
fig.tight_layout()
plt.show()

---
## 5. MC Convergence

Is the d' bump real, or just Monte Carlo sampling noise?  
Run with increasing n_mc and check if the bump persists.

In [ ]:
# Run the ISI=[1] L=15 sweep with different n_mc values

mc_values = [8, 32, 128, 256]
mc_sweeps = {}

for n_mc in mc_values:
    print(f"\n--- n_mc={n_mc} ---")
    mc_sweeps[n_mc] = sweep_sigma1_decomposed(
        e_short, sigma0=SIGMA0, sigma1_grid=SIGMA1_GRID,
        n_mc=n_mc, desc=f'n_mc={n_mc}', seed=900_000,
    )

# Plot d' curves for each n_mc
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

for n_mc in mc_values:
    results = mc_sweeps[n_mc]
    s1 = np.array([r['sigma1'] for r in results])
    dp = np.array([r['dprime'] for r in results])
    ax.plot(s1, dp, 'o-', ms=3, label=f'n_mc={n_mc}')

ax.axhline(sigma1_human.get(1, 0), ls=':', color='red', alpha=0.7, label="human d'")
ax.set_xscale('log')
ax.set_xlabel('sigma1')
ax.set_ylabel("d'")
ax.set_title("Section 5: MC convergence for ISI=[1] L=15\n"
             "Does the bump persist with more MC samples?")
ax.legend(fontsize=9, frameon=False)
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

---
## 6. Bank Size Controlled Experiment

The key confound: trial position = bank size. Early in the sequence the bank
has few items, late it has many. The FA score = min distance over the bank,
which is a **decreasing function of bank size** (order statistics).

Test: generate sequences of varying length, all with ISI=1 only. If the
bump flattens at longer sequences, bank size is the mechanism.

In [ ]:
# Generate ISI=[1]-only sequences at different lengths

bank_configs = {}
for L in [15, 30, 45, 60, 90]:
    try:
        e, k = make_compact_multi_isi_sequences(
            stimulus_pool=stimulus_pool, isi_values=[1],
            n_sequences=10, length=L, min_pairs_per_isi=5, seed=5000 + L,
        )
        bank_configs[L] = e
        print(f"L={L}: {len(e)} seqs x {len(e[0])} trials")
    except Exception as ex:
        print(f"L={L}: FAILED ({ex})")

# Run sigma1 sweep for each
bank_sweeps = {}
for L, exps in bank_configs.items():
    print(f"\n--- L={L} ---")
    bank_sweeps[L] = sweep_sigma1_decomposed(
        exps, sigma0=SIGMA0, sigma1_grid=SIGMA1_GRID,
        n_mc=N_MC, desc=f'L={L}', seed=1_000_000 + L,
    )

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
cmap = plt.cm.viridis
colors = cmap(np.linspace(0.05, 0.95, len(bank_sweeps)))

for color, (L, results) in zip(colors, sorted(bank_sweeps.items())):
    s1 = np.array([r['sigma1'] for r in results])
    dp = np.array([r['dprime'] for r in results])
    hm = np.array([r['hit_mean'] for r in results])
    fm = np.array([r['fa_mean'] for r in results])

    axes[0].plot(s1, dp, 'o-', ms=3, color=color, label=f'L={L}')
    axes[1].plot(s1, hm, 'o-', ms=3, color=color, label=f'L={L}')
    axes[2].plot(s1, fm, 'o-', ms=3, color=color, label=f'L={L}')

axes[0].axhline(sigma1_human.get(1, 0), ls=':', color='red', alpha=0.7)

for ax, title in zip(axes, ["d'(ISI=1)", "Mean hit score", "Mean FA score"]):
    ax.set_xscale('log')
    ax.set_xlabel('sigma1')
    ax.set_title(title)
    ax.legend(fontsize=8, frameon=False)
    ax.grid(alpha=0.25)

fig.suptitle("Section 6: Bank-size controlled experiment\n"
             "If bump flattens at longer L, the FA order-statistics effect is the cause",
             y=1.05, fontsize=13)
fig.tight_layout()
plt.show()

---
## 7. Stimulus Diversity

`make_compact_multi_isi_sequences` pre-truncates the stimulus pool to
`length // 3 * 2` items. For L=15, that's only **10 stimuli**. With
`stimulus_rotation_step = L // (3 * n_isi)`, sequences rotate through the
same tiny pool with very few distinct orderings.

**Hypothesis**: Low stimulus diversity means FA comparisons always involve the
same small set of inter-stimulus distances, inflating FA variance at low sigma1
and biasing d'.

**Test**: Pass the FULL stimulus pool to `make_compact_multi_isi_sequences`
(bypassing the truncation) and see if the non-monotonicity changes.

In [ ]:
# Quantify the stimulus diversity problem

print("Stimulus diversity across sequence configs:")
print(f"{'Config':<25s} {'Pool size':>10s} {'Unique/seq':>12s} {'Total unique':>13s} {'Overlap %':>10s}")
print('-' * 75)

for name, cfg in seq_configs.items():
    exps = cfg['exps']
    unique_per_seq = [len(set(seq)) for seq in exps]
    all_unique = set()
    for seq in exps:
        all_unique.update(seq)

    # Pairwise overlap: what fraction of stimuli are shared between sequences?
    overlaps = []
    for i in range(len(exps)):
        for j in range(i + 1, len(exps)):
            s_i = set(exps[i])
            s_j = set(exps[j])
            overlap = len(s_i & s_j) / len(s_i | s_j)
            overlaps.append(overlap)

    pool_size = len(all_unique)
    print(f"{name:<25s} {pool_size:>10d} {np.mean(unique_per_seq):>12.1f} "
          f"{len(all_unique):>13d} {np.mean(overlaps)*100:>9.1f}%")

print(f"\nFull stimulus pool: {len(stimulus_pool)} items")
print(f"\nFor L=15: pool = {15 // 3 * 2} stimuli, rotation_step = {15 // 3}")
print(f"For L=60: pool = {60 // 3 * 2} stimuli, rotation_step = {60 // 3}")

In [ ]:
# HIGH-DIVERSITY sequences: use the full pool, ensure each sequence draws
# fresh stimuli by using a different seed offset and the full pool

from utils.sequence_utils import ISISequence, StimulusManager

def make_high_diversity_sequences(
    stimulus_pool, isi_values, n_sequences, length, min_pairs_per_isi=5, seed=42,
):
    """Like make_compact_multi_isi_sequences but each sequence draws from a
    DIFFERENT slice of the full stimulus pool (no overlap between seqs)."""
    isi_values = list(isi_values)
    isi_conditions = [-1] + isi_values
    unique_per_seq = length // 3 * 2  # stimuli needed per sequence
    total_needed = unique_per_seq * n_sequences

    if len(stimulus_pool) < total_needed:
        print(f"WARNING: pool ({len(stimulus_pool)}) < needed ({total_needed}), "
              f"some overlap will occur")

    # Stage 1: generate ISI patterns (same as original)
    isi_seq = ISISequence(length=length, isi_values=isi_conditions, seed=seed)
    isi_seq.generate_n(n=n_sequences, min_pairs_per_isi=min_pairs_per_isi)

    # Stage 2: assign DIFFERENT stimuli to each sequence
    rng = random.Random(seed + 99)
    pool = list(stimulus_pool)
    rng.shuffle(pool)

    experiment_list = []
    isi_keys = []

    for j in range(n_sequences):
        seq, pairs = isi_seq.get_sequence_and_isi_pairings(j)

        # Each sequence gets its own slice of the pool
        start = (j * unique_per_seq) % len(pool)
        end = start + unique_per_seq
        if end <= len(pool):
            seq_pool = pool[start:end]
        else:
            # Wrap around
            seq_pool = pool[start:] + pool[:end - len(pool)]

        # Use StimulusManager for a single sequence with this unique pool
        sm = StimulusManager(
            stimulus_ids=seq_pool,
            isi_values=isi_conditions,
            length=length,
            seed=seed + j * 1000,
            shuffle=True,
        )
        sm.get_assignments_from_pairs(pairs, seq=seq)
        experiment_list.append(sm.assignments[0])
        isi_keys.append(sm.seqs[0])

    return experiment_list, isi_keys


# Generate high-diversity versions of the same configs
e_short_hd, k_short_hd = make_high_diversity_sequences(
    stimulus_pool, isi_values=[1],
    n_sequences=8, length=15, min_pairs_per_isi=5, seed=2000,
)

e_long_hd, k_long_hd = make_high_diversity_sequences(
    stimulus_pool, isi_values=[1],
    n_sequences=10, length=60, min_pairs_per_isi=5, seed=2001,
)

# Check diversity
for label, exps in [("LOW div L=15", e_short), ("HIGH div L=15", e_short_hd),
                    ("LOW div L=60", e_long), ("HIGH div L=60", e_long_hd)]:
    all_stim = set()
    for seq in exps:
        all_stim.update(seq)
    overlaps = []
    for i in range(len(exps)):
        for j in range(i + 1, len(exps)):
            overlap = len(set(exps[i]) & set(exps[j])) / len(set(exps[i]) | set(exps[j]))
            overlaps.append(overlap)
    print(f"{label:<20s}: {len(all_stim):>4d} unique stimuli, "
          f"{np.mean(overlaps)*100:.1f}% avg pairwise overlap")

In [ ]:
# Run sigma1 sweep: LOW diversity vs HIGH diversity

print("--- LOW diversity L=15 (already computed) ---")
# sweeps['ISI=[1] L=15'] already exists

print("\n--- HIGH diversity L=15 ---")
sweep_hd_short = sweep_sigma1_decomposed(
    e_short_hd, sigma0=SIGMA0, sigma1_grid=SIGMA1_GRID,
    n_mc=N_MC, desc='HIGH div L=15', seed=1_100_000,
)

print("\n--- HIGH diversity L=60 ---")
sweep_hd_long = sweep_sigma1_decomposed(
    e_long_hd, sigma0=SIGMA0, sigma1_grid=SIGMA1_GRID,
    n_mc=N_MC, desc='HIGH div L=60', seed=1_200_000,
)

# Plot comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

comparisons = [
    ('ISI=[1] L=15 (low div)', sweeps['ISI=[1] L=15'], 'tab:blue', '-'),
    ('ISI=[1] L=15 (HIGH div)', sweep_hd_short, 'tab:blue', '--'),
    ('ISI=[1] L=60 (low div)', sweeps['ISI=[1] L=60'], 'tab:orange', '-'),
    ('ISI=[1] L=60 (HIGH div)', sweep_hd_long, 'tab:orange', '--'),
]

for name, results, color, ls in comparisons:
    s1 = np.array([r['sigma1'] for r in results])
    dp = np.array([r['dprime'] for r in results])
    hm = np.array([r['hit_mean'] for r in results])
    fm = np.array([r['fa_mean'] for r in results])
    hs = np.array([r['hit_std'] for r in results])
    fs = np.array([r['fa_std'] for r in results])

    axes[0, 0].plot(s1, dp, 'o'+ls, ms=3, color=color, label=name)
    axes[0, 1].plot(s1, hm, 'o'+ls, ms=3, color=color, label=name)
    axes[0, 2].plot(s1, fm, 'o'+ls, ms=3, color=color, label=name)
    axes[1, 0].plot(s1, fm - hm, 'o'+ls, ms=3, color=color, label=name)
    axes[1, 1].plot(s1, hs, 'o'+ls, ms=3, color=color, label=name)
    axes[1, 2].plot(s1, fs, 'o'+ls, ms=3, color=color, label=name)

titles = ["d'(ISI=1)", "Mean hit score", "Mean FA score",
          "Gap (FA - hit)", "Hit std", "FA std"]
for ax, title in zip(axes.flat, titles):
    ax.set_xscale('log')
    ax.set_xlabel('sigma1')
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=6, frameon=False)
    ax.grid(alpha=0.25)

fig.suptitle("Section 7: Effect of stimulus diversity on the non-monotonicity\n"
             "Dashed = HIGH diversity (unique stimuli per sequence), "
             "Solid = LOW diversity (original, shared pool)",
             y=1.05, fontsize=13)
fig.tight_layout()
plt.show()

---
## 8. Why Does FA std Have a U-Shape?

The FA score = min distance from a novel probe to ALL drifted memories.

**Three regimes of FA std behavior:**

1. **Low sigma1**: Memories are accurate. FA std reflects the **natural variance
   of nearest-neighbor distances** in the embedding space. Some foils happen to
   be close to a stored stimulus (low FA), others far (high FA). The spread
   comes from the stimulus geometry.

2. **Moderate sigma1**: Drift noise starts to dominate over inter-stimulus
   distance differences. Each memory jiggles randomly, **homogenizing** the
   nearest-neighbor distances across foils. FA std **decreases**.

3. **High sigma1**: Drift noise is so large that memory positions are essentially
   random. The min-distance distribution is now driven by noise magnitude, which
   has high variance. FA std **increases**.

The minimum of FA std is the crossover from "stimulus-geometry-dominated" to
"noise-dominated" variability.

**Test**: If we remove the min() aggregation and look at the mean distance to
all memories (not just the nearest), the U-shape should vanish — it's specific
to the order-statistics (min) operation.

In [ ]:
# Analytical demonstration: std of min(X1, ..., Xk) for Gaussian Xi
# As noise (spread) increases, the min-of-k distribution first compresses then expands

from scipy.stats import norm

def min_of_k_gaussian_stats(k, mu_values, sigma_noise, n_samples=50_000):
    """Simulate: k items at positions mu_values, each with noise sigma_noise.
    For each sample, compute min distance from origin to any noisy item.
    Return mean and std of the min-distance distribution.
    """
    rng = np.random.default_rng(42)
    mu_arr = np.array(mu_values).reshape(1, -1)  # (1, k)

    # Noisy positions: (n_samples, k)
    noisy = mu_arr + rng.normal(0, sigma_noise, size=(n_samples, k))

    # Distance from origin (the "probe") to each noisy memory
    distances = np.abs(noisy)  # 1D for simplicity

    # FA-like score: min distance
    min_dists = distances.min(axis=1)
    # Mean distance (no min)
    mean_dists = distances.mean(axis=1)

    return {
        'min_mean': np.mean(min_dists), 'min_std': np.std(min_dists),
        'avg_mean': np.mean(mean_dists), 'avg_std': np.std(mean_dists),
    }


# Simulate with k=5 items at fixed positions (like a small bank)
k = 5
mu_values = [0.5, 1.2, 2.0, 3.5, 5.0]  # "natural" distances from probe
sigma_range = np.exp(np.linspace(np.log(0.01), np.log(10.0), 40))

min_stds, avg_stds, min_means, avg_means = [], [], [], []
for sig in sigma_range:
    stats = min_of_k_gaussian_stats(k, mu_values, sig)
    min_stds.append(stats['min_std'])
    avg_stds.append(stats['avg_std'])
    min_means.append(stats['min_mean'])
    avg_means.append(stats['avg_mean'])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(sigma_range, min_stds, 'o-', ms=3, color='tab:blue', label='std of min(distances)')
axes[0].plot(sigma_range, avg_stds, 's-', ms=3, color='tab:orange', label='std of mean(distances)')
axes[0].set_title('Std of score vs noise level')
axes[0].set_xlabel('sigma (noise)')
axes[0].set_ylabel('Std of score distribution')
axes[0].legend(fontsize=9, frameon=False)
axes[0].set_xscale('log')
axes[0].grid(alpha=0.25)

axes[1].plot(sigma_range, min_means, 'o-', ms=3, color='tab:blue', label='mean of min(distances)')
axes[1].plot(sigma_range, avg_means, 's-', ms=3, color='tab:orange', label='mean of mean(distances)')
axes[1].set_title('Mean of score vs noise level')
axes[1].set_xlabel('sigma (noise)')
axes[1].set_ylabel('Mean score')
axes[1].legend(fontsize=9, frameon=False)
axes[1].set_xscale('log')
axes[1].grid(alpha=0.25)

# Now do it for different k values
axes[2].set_title('FA std U-shape depends on bank size k')
for k_val in [2, 5, 10, 20]:
    stds = []
    for sig in sigma_range:
        # Random mu values to simulate realistic inter-stimulus distances
        rng_mu = np.random.default_rng(123)
        mus = rng_mu.exponential(2.0, size=k_val)
        stats = min_of_k_gaussian_stats(k_val, mus.tolist(), sig)
        stds.append(stats['min_std'])
    axes[2].plot(sigma_range, stds, 'o-', ms=3, label=f'k={k_val}')

axes[2].set_xlabel('sigma (noise)')
axes[2].set_ylabel('Std of min-distance')
axes[2].legend(fontsize=9, frameon=False)
axes[2].set_xscale('log')
axes[2].grid(alpha=0.25)

fig.suptitle("Section 8: Why FA std has a U-shape\n"
             "The min() operation creates a non-monotonic std; mean() does not.\n"
             "Larger bank (k) shifts the minimum leftward.",
             y=1.07, fontsize=13)
fig.tight_layout()
plt.show()

print("\nKey insight: the U-shape in FA std is an intrinsic property of the")
print("min-of-k order statistic. At low noise, std reflects stimulus geometry.")
print("At moderate noise, drift homogenizes distances → std drops.")
print("At high noise, noise variance dominates → std rises.")

---
## Summary

**Fill in after running:**

1. **d' decomposition** (Section 1): The non-monotonicity is driven by
   [hit scores / FA scores / both]. Specifically, [...] changes non-monotonically.

2. **Score distributions** (Section 2): At the d' peak, the hit/FA separation is
   [larger / smaller / same] compared to low sigma1.

3. **Noise schedule** (Section 3): For ISI=1, sigma1 controls one round of
   drift at age=2. Encoding noise sigma0 [dominates / is comparable to] drift.

4. **Target vs non-target** (Section 4): The transition to non-target dominance
   occurs at sigma1 ≈ [___].

5. **MC convergence** (Section 5): The bump [persists / vanishes] with more MC samples.

6. **Bank size** (Section 6): The bump [does / does not] flatten at longer sequences.

7. **Stimulus diversity** (Section 7): With high diversity, the non-monotonicity
   [persists / disappears / changes shape]. FA mean [increases / decreases / stays same].
   FA std [changes / stays same].

8. **FA std U-shape** (Section 8): Confirmed to be an intrinsic property of the
   min-of-k order statistic. [Does / does not] match the analytical prediction.

**Conclusion**: The non-monotonic d'(ISI=1) vs sigma1 curve is caused by [___].